# SIM Intervention Demand Forecasting

**YCIT 440 – AI and Data Science Capstone Project**

**Team 4**

## 2. Baseline Forecasting

### 2.1. Project Objective

The objective of this stage is to develop a simple forecasting baseline for the total number of recorded daily interventions in SIM operational Divisions 1 through 6.

The baseline will use the average of recent historical intervention counts for the same day of the week.

For each forecast day D, the prediction will be prepared on D−1 using intervention counts available through D−3.

Different historical averaging windows will be compared using chronological validation data. The selected baseline will serve as the reference for evaluating the more complex forecasting models.

Mean Absolute Error (MAE) will be the primary performance metric, consistent with the revised Business Problem Framing Report.

### 2.2. Import Required Libraries

We will use pandas to load and organize the daily intervention data and NumPy for numerical operations.

Additional libraries will be introduced when needed for baseline evaluation.

In [ ]:

import pandas as pd
import numpy as np


### 2.3. Load the Prepared Daily Dataset

The daily intervention dataset was created and validated during Stage 1: Modeling Dataset — Data Preparation.

It contains one observation for each operational division and calendar day, including the six missing intervention counts for December 31, 2024.

We will load this prepared dataset and verify its dimensions, date range, and missing target values before implementing the forecasting baseline.

In [ ]:

# Load the prepared daily intervention dataset
input_file = "../../data/interim/modelling/SIM_Daily_Interventions.csv"

df_daily = pd.read_csv(input_file, parse_dates=["Date"])

# Display dataset information
print("Dataset dimensions:", df_daily.shape)
print("First date:", df_daily["Date"].min())
print("Last date:", df_daily["Date"].max())
print("Missing target values:", df_daily["Intervention_Count"].isna().sum())

# Display the first six observations
display(df_daily.head(6))


Dataset dimensions: (14754, 3)
First date: 2020-01-01 00:00:00
Last date: 2026-09-24 00:00:00
Missing target values: 6


,Date,DIVISION,Intervention_Count
0,2020-01-01,1,35.0
1,2020-01-01,2,67.0
2,2020-01-01,3,55.0
3,2020-01-01,4,70.0
4,2020-01-01,5,64.0
5,2020-01-01,6,69.0


### 2.4. Define the Baseline Forecasting Logic

The baseline will forecast daily intervention counts using the average of recent observations from the same day of the week, calculated separately for each operational division.

For each forecast date D, only historical intervention counts through D−3 will be available. The baseline will therefore use eligible observations from previous matching weekdays without accessing future information.

The number of historical observations used in the average will be selected through chronological validation. For initial implementation and verification, we will use a four-observation window as an illustrative example, not as the final baseline configuration.

In [ ]:

def predict_same_weekday_average(data, division, forecast_date, window=4):
    # Convert the forecast date to datetime
    forecast_date = pd.Timestamp(forecast_date)

    # Calculate the latest allowed historical date
    cutoff_date = forecast_date - pd.Timedelta(days=3)

    # Select records from the requested division
    division_data = data[data["DIVISION"] == division]

    # Keep historical records through the D-3 cutoff
    historical_data = division_data[
        division_data["Date"] <= cutoff_date
    ]

    # Keep observations from the same weekday as the forecast
    same_weekday = historical_data[
        historical_data["Date"].dt.dayofweek == forecast_date.dayofweek
    ]

    # Exclude observations with missing intervention counts
    same_weekday = same_weekday.dropna(subset=["Intervention_Count"])

    # Select the most recent matching weekdays
    recent_observations = same_weekday.sort_values("Date").tail(window)

    # Require the full number of historical observations
    if len(recent_observations) < window:
        return np.nan

    # Calculate the average intervention count
    prediction = recent_observations["Intervention_Count"].mean()

    return prediction


### 2.5. Test the Baseline Function

We will test the baseline function using Division 1 and January 19, 2024, as an example forecast date.

This test uses a four-observation window to verify that the prediction is calculated from earlier matching weekdays and respects the D−3 cutoff.

The example is used to verify the calculation, not to evaluate forecasting performance.

In [ ]:

# Define an example forecast
example_division = 1
example_date = "2024-01-19"

# Calculate the baseline prediction
example_prediction = predict_same_weekday_average(
    data=df_daily,
    division=example_division,
    forecast_date=example_date,
    window=4
)

print("Division:", example_division)
print("Forecast date:", example_date)
print("Baseline prediction:", example_prediction)


Division: 1
Forecast date: 2024-01-19
Baseline prediction: 30.0


### 2.6. Verify Historical Observations Used in the Baseline

We will manually verify the historical observations used to calculate the example baseline prediction.

For January 19, 2024, the latest permitted historical date is January 16, 2024 (D−3).

We will identify the four most recent valid observations from the same weekday and calculate their average to verify that the baseline function produces the expected result.

In [ ]:

# Define the forecast date and information cutoff
forecast_date = pd.Timestamp("2024-01-19")
cutoff_date = forecast_date - pd.Timedelta(days=3)

# Select Division 1 records
division_data = df_daily[df_daily["DIVISION"] == 1]

# Select eligible historical records
historical_data = division_data[
    (division_data["Date"] <= cutoff_date) &
    (division_data["Date"].dt.dayofweek == forecast_date.dayofweek)
]

# Select the four most recent valid observations
recent_observations = (
    historical_data.dropna(subset=["Intervention_Count"])
    .sort_values("Date")
    .tail(4)
)

# Display the selected observations
print("Forecast date:", forecast_date.date())
print("Cutoff date:", cutoff_date.date())

print("\nHistorical observations used:")
display(recent_observations)

# Calculate the average manually
manual_prediction = recent_observations["Intervention_Count"].mean()

print("Manually calculated prediction:", manual_prediction)
print("Function prediction:", example_prediction)
print("Predictions match:", np.isclose(manual_prediction, example_prediction))


Forecast date: 2024-01-19
Cutoff date: 2024-01-16

Historical observations used:


,Date,DIVISION,Intervention_Count
8706,2023-12-22,1,33.0
8748,2023-12-29,1,27.0
8790,2024-01-05,1,30.0
8832,2024-01-12,1,30.0


Manually calculated prediction: 30.0
Function prediction: 30.0
Predictions match: True


**Methodological Observation**

The baseline function was verified using Division 1 and January 19, 2024, as an example forecast.

The prediction was calculated from the four most recent valid observations for the same weekday, all occurring before the D−3 information cutoff. The historical counts were 33, 27, 30, and 30 interventions, producing an average of 30.0.

The manual calculation matched the function output, confirming that the baseline logic works correctly for this example and respects the defined information cutoff.

This verification confirms the calculation procedure, but does not establish predictive accuracy. Forecast performance will be assessed through chronological validation.

### 2.7. Generate Baseline Predictions for a Sample Period

We will extend the baseline function to generate predictions for all six operational divisions over a sample period in January 2024.

Each forecast will use a four-observation same-weekday average and historical intervention counts available through D−3.

This initial test will verify that the forecasting procedure can be applied consistently across multiple dates and divisions before implementing the full chronological validation.

In [ ]:

# Define the sample forecasting period
start_date = "2024-01-01"
end_date = "2024-01-31"

# Select dates from the sample period
forecast_dates = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

# Create an empty list to store predictions
baseline_results = []

# Generate predictions for each date and division
for forecast_date in forecast_dates:
    for division in [1, 2, 3, 4, 5, 6]:

        prediction = predict_same_weekday_average(
            data=df_daily,
            division=division,
            forecast_date=forecast_date,
            window=4
        )

        baseline_results.append({
            "Date": forecast_date,
            "DIVISION": division,
            "Baseline_Prediction": prediction
        })

# Convert the predictions into a DataFrame
df_baseline_sample = pd.DataFrame(baseline_results)

# Display results
print("Baseline predictions:", df_baseline_sample.shape)
print(
    "Missing predictions:",
    df_baseline_sample["Baseline_Prediction"].isna().sum()
)

display(df_baseline_sample.head(12))


Baseline predictions: (186, 3)
Missing predictions: 0


,Date,DIVISION,Baseline_Prediction
0,2024-01-01,1,34.50
1,2024-01-01,2,101.25
2,2024-01-01,3,88.75
3,2024-01-01,4,73.00
4,2024-01-01,5,58.50
5,2024-01-01,6,50.50
6,2024-01-02,1,35.50
7,2024-01-02,2,61.25
8,2024-01-02,3,68.25
9,2024-01-02,4,57.25


**Methodological Observation**

The baseline successfully generated 186 predictions for January 2024, covering all six operational divisions and 31 calendar days.

No missing predictions were identified. The results confirm that the same-weekday averaging procedure can be applied across multiple dates and divisions using the four-observation example window.

The predictions were retained as decimal values to preserve the original averages for subsequent error calculations.

This test verifies the generation of predictions but does not establish forecasting accuracy. The final baseline window will be selected through chronological validation.

### 2.8. Match Baseline Predictions with Actual Intervention Counts

We will combine the baseline predictions with the corresponding actual daily intervention counts.

The datasets will be matched using `Date` and `DIVISION`, so each prediction is associated with the correct observed intervention count.

The actual counts will be used only to evaluate the predictions, not to generate them. This distinction is important to prevent data leakage.

Before calculating forecast errors, we will verify that all sample predictions have a corresponding actual value.

In [ ]:

# Combine baseline predictions with actual intervention counts
df_baseline_comparison = pd.merge(
    df_baseline_sample,
    df_daily,
    on=["Date", "DIVISION"],
    how="left"
)

# Display the combined dataset
print("Comparison dataset:", df_baseline_comparison.shape)

# Check for missing actual intervention counts
missing_actuals = df_baseline_comparison["Intervention_Count"].isna().sum()

print("Missing actual counts:", missing_actuals)

# Display the first 12 observations
display(df_baseline_comparison.head(12))


Comparison dataset: (186, 4)
Missing actual counts: 0


,Date,DIVISION,Baseline_Prediction,Intervention_Count
0,2024-01-01,1,34.50,27.0
1,2024-01-01,2,101.25,34.0
2,2024-01-01,3,88.75,72.0
3,2024-01-01,4,73.00,61.0
4,2024-01-01,5,58.50,65.0
5,2024-01-01,6,50.50,46.0
6,2024-01-02,1,35.50,33.0
7,2024-01-02,2,61.25,47.0
8,2024-01-02,3,68.25,55.0
9,2024-01-02,4,57.25,60.0


**Methodological Observation**

All 186 baseline predictions for January 2024 were successfully matched with their corresponding actual intervention counts. No missing actual values were identified.

The comparison dataset is ready for forecast error calculations. Actual intervention counts were added only after generating the predictions, ensuring that they were not used as inputs to the baseline forecasting procedure.

Differences between predicted and actual counts are visible across individual observations. However, individual errors are not sufficient to evaluate overall forecasting performance.

### 2.9. Calculate Forecast Errors

We will calculate the difference between each baseline prediction and the corresponding actual intervention count.

Forecast error will be defined as predicted interventions minus actual interventions, consistent with the revised Business Problem Framing Report.

We will also calculate the absolute error to measure the magnitude of each prediction error, regardless of whether the baseline overestimated or underestimated intervention demand.

These values will be used to evaluate baseline performance during chronological validation.

In [ ]:

# Calculate forecast errors
df_baseline_comparison["Error"] = (
    df_baseline_comparison["Baseline_Prediction"]
    - df_baseline_comparison["Intervention_Count"]
)

# Calculate absolute errors
df_baseline_comparison["Absolute_Error"] = (
    df_baseline_comparison["Error"].abs()
)

# Display the first 12 observations
display(df_baseline_comparison.head(12))


,Date,DIVISION,Baseline_Prediction,Intervention_Count,Error,Absolute_Error
0,2024-01-01,1,34.50,27.0,7.50,7.50
1,2024-01-01,2,101.25,34.0,67.25,67.25
2,2024-01-01,3,88.75,72.0,16.75,16.75
3,2024-01-01,4,73.00,61.0,12.00,12.00
4,2024-01-01,5,58.50,65.0,-6.50,6.50
5,2024-01-01,6,50.50,46.0,4.50,4.50
6,2024-01-02,1,35.50,33.0,2.50,2.50
7,2024-01-02,2,61.25,47.0,14.25,14.25
8,2024-01-02,3,68.25,55.0,13.25,13.25
9,2024-01-02,4,57.25,60.0,-2.75,2.75


**Methodological Observation**

Forecast errors were calculated for the January 2024 baseline predictions using the difference between predicted and actual intervention counts.

Positive errors indicate overprediction, while negative errors indicate underprediction. Absolute errors represent the magnitude of each difference without considering its direction.

These calculations provide the basis for evaluating prediction accuracy using Mean Absolute Error (MAE) and systematic overprediction or underprediction using Mean Error (bias).

### 2.10. Calculate Baseline MAE and Mean Error

We will calculate Mean Absolute Error (MAE) and Mean Error (bias) for the January 2024 baseline predictions.

MAE measures the average magnitude of forecasting errors, while Mean Error indicates whether the baseline tends to overpredict or underpredict intervention counts.

These metrics will first be calculated across all six divisions as an initial implementation check. Division-level performance will be examined separately.

The results are preliminary because the four-observation window has not yet been selected through chronological validation.

In [ ]:

# Calculate Mean Absolute Error
baseline_mae = df_baseline_comparison["Absolute_Error"].mean()

# Calculate Mean Error (bias)
baseline_bias = df_baseline_comparison["Error"].mean()

# Display the results
print("Evaluation period: January 2024")
print("Baseline window: 4 same-weekday observations")
print("Number of predictions:", len(df_baseline_comparison))

print("\nMean Absolute Error (MAE):", round(baseline_mae, 2))
print("Mean Error (Bias):", round(baseline_bias, 2))


Evaluation period: January 2024
Baseline window: 4 same-weekday observations
Number of predictions: 186

Mean Absolute Error (MAE): 7.08
Mean Error (Bias): 2.25


**Methodological Observation**

The four-observation same-weekday baseline achieved a Mean Absolute Error (MAE) of 7.08 interventions and a Mean Error (bias) of +2.25 interventions across 186 predictions in January 2024.

The positive bias indicates that the baseline overestimated intervention counts on average during this sample period.

These results provide an initial evaluation of the baseline implementation. However, they do not establish its overall forecasting performance because the analysis covers only one month and uses an illustrative window that has not yet been selected through chronological validation.

Performance will also be examined by division to identify differences that may be hidden by the overall averages.

### 2.11. Calculate MAE and Bias by Division

We will calculate Mean Absolute Error (MAE) and Mean Error (bias) separately for each operational division.

This analysis will help identify whether the baseline performs differently across the six divisions and whether some divisions show a greater tendency toward overprediction or underprediction.

The results will remain preliminary because they are based on January 2024 and an illustrative four-observation baseline window.

In [ ]:

# Calculate average errors for each division
division_metrics = df_baseline_comparison.groupby("DIVISION")[
    ["Absolute_Error", "Error"]
].mean()

# Rename columns to match the evaluation metrics
division_metrics = division_metrics.rename(
    columns={
        "Absolute_Error": "MAE",
        "Error": "Bias"
    }
)

# Round the metrics for presentation
division_metrics = division_metrics.round(2)

# Display the results
print("Baseline performance by division - January 2024:")
display(division_metrics)


Baseline performance by division - January 2024:


,MAE,Bias
DIVISION,,
1,4.99,2.67
2,9.85,3.99
3,7.88,4.02
4,6.83,1.20
5,6.24,0.66
6,6.72,0.96


**Methodological Observation**

The January 2024 baseline evaluation shows differences in forecasting performance across the six operational divisions.

Division 2 had the highest Mean Absolute Error (9.85 interventions), while Division 1 had the lowest (4.99 interventions).

All six divisions showed positive Mean Error values, indicating average overprediction during the sample period. The largest positive biases were observed in Divisions 3 (+4.02) and 2 (+3.99).

These findings are preliminary and should not be interpreted as evidence of consistent performance differences across divisions. The results may be influenced by differences in intervention volume and the limited evaluation period.

A broader chronological validation will be required before selecting the baseline configuration.

### 2.12. Define Candidate Baseline Windows

The initial baseline implementation used four historical observations from the same weekday as an illustrative example.

We will now compare several candidate window sizes to determine how the amount of historical information affects forecasting accuracy.

The candidate windows will contain 4, 8, 13, and 26 valid observations from the same weekday. These values represent approximately 4, 8, 13, and 26 weeks of historical data when no observations are missing.

The final window will be selected using chronological validation from 2021 through 2024, based on Mean Absolute Error (MAE). The reserved evaluation period from January 1, 2025, through September 24, 2026, will not be used for window selection.

In [ ]:

# Define candidate baseline windows
candidate_windows = [4, 8, 13, 26]

# Display the candidate windows
print("Candidate baseline windows:")

for window in candidate_windows:
    print(window, "same-weekday observations")


Candidate baseline windows:
4 same-weekday observations
8 same-weekday observations
13 same-weekday observations
26 same-weekday observations


### 2.13. Verify Historical Data Availability

Before implementing chronological validation, we will verify that the available historical records support all candidate baseline windows at the beginning of the validation period.

We will test January 1, 2021, for each operational division using the largest candidate window of 26 same-weekday observations.

This check will confirm whether the baseline can generate predictions from the start of the proposed validation period while respecting the D−3 information cutoff.

In [ ]:

# Define the first validation date
first_validation_date = "2021-01-01"

# Select the largest candidate window
largest_window = max(candidate_windows)

# Test the baseline for each division
for division in [1, 2, 3, 4, 5, 6]:

    prediction = predict_same_weekday_average(
        data=df_daily,
        division=division,
        forecast_date=first_validation_date,
        window=largest_window
    )

    print(
        "Division:", division,
        "| Window:", largest_window,
        "| Prediction:", prediction
    )


Division: 1 | Window: 26 | Prediction: 23.692307692307693
Division: 2 | Window: 26 | Prediction: 50.23076923076923
Division: 3 | Window: 26 | Prediction: 52.88461538461539
Division: 4 | Window: 26 | Prediction: 55.38461538461539
Division: 5 | Window: 26 | Prediction: 54.53846153846154
Division: 6 | Window: 26 | Prediction: 33.5


**Methodological Observation**

All six operational divisions produced valid baseline predictions for January 1, 2021, using the largest candidate window of 26 same-weekday observations.

This confirms that sufficient historical data are available at the beginning of the chronological validation period to support all four candidate windows (4, 8, 13, and 26 observations).

The baseline can therefore be evaluated from January 1, 2021, without requiring an additional initial period for historical data collection.

### 2.14. Generate Chronological Validation Predictions

We will generate baseline forecasts for operational Divisions 1 through 6 over the chronological validation period from January 1, 2021, through December 31, 2024.

Each candidate window (4, 8, 13, and 26 same-weekday observations) will be evaluated using the same forecast dates and the same D−3 information cutoff.

The missing intervention counts for December 31, 2024, will remain unavailable for error calculation.

This procedure will allow us to compare candidate baseline windows using the same chronological validation period without using records from the reserved final evaluation period.

**Original Implementation — Retained for Learning Purposes**

The original implementation used nested loops to generate 35,064 baseline predictions. Although the results were correct, execution took approximately two minutes.

This code has been commented out to prevent accidental execution. It is retained for educational purposes and to document the progression from the initial implementation to the optimized version.

The optimized implementation reduced execution time to approximately one second and matched the original results in all 144 verification cases.

In [ ]:

# # Define the chronological validation period
# validation_dates = pd.date_range(
#     start="2021-01-01",
#     end="2024-12-31",
#     freq="D"
# )

# # Create an empty list to store validation predictions
# validation_results = []

# # Generate predictions for each candidate window
# for window in candidate_windows:

#     for forecast_date in validation_dates:

#         for division in [1, 2, 3, 4, 5, 6]:

#             prediction = predict_same_weekday_average(
#                 data=df_daily,
#                 division=division,
#                 forecast_date=forecast_date,
#                 window=window
#             )

#             validation_results.append({
#                 "Date": forecast_date,
#                 "DIVISION": division,
#                 "Window": window,
#                 "Baseline_Prediction": prediction
#             })

# # Convert the results into a DataFrame
# df_validation = pd.DataFrame(validation_results)

# # Display the results
# print("Validation predictions:", df_validation.shape)

# print(
#     "Missing predictions:",
#     df_validation["Baseline_Prediction"].isna().sum()
# )

# display(df_validation.head(12))


### 2.14. Generate Chronological Validation Predictions — Optimized Implementation

The initial implementation used nested loops to generate predictions for every combination of forecast date, division, and candidate window.

To improve execution efficiency, we will use pandas grouping, rolling averages, and date-based matching.

The optimized implementation preserves the original baseline methodology: each prediction uses the most recent valid same-weekday observations for the corresponding division, with historical intervention counts restricted to D−3.

The four candidate windows and the chronological validation period remain unchanged.

In [ ]:

# Define the chronological validation period
validation_dates = pd.date_range(
    start="2021-01-01",
    end="2024-12-31",
    freq="D"
)

# Create all combinations of validation dates and divisions
validation_calendar = pd.MultiIndex.from_product(
    [validation_dates, [1, 2, 3, 4, 5, 6]],
    names=["Date", "DIVISION"]
).to_frame(index=False)

# Add the weekday and the D-3 information cutoff
validation_calendar["Weekday"] = validation_calendar["Date"].dt.dayofweek
validation_calendar["Cutoff_Date"] = (
    validation_calendar["Date"] - pd.Timedelta(days=3)
)

# Prepare historical records with valid intervention counts
historical_data = df_daily.dropna(
    subset=["Intervention_Count"]
).copy()

historical_data["Weekday"] = historical_data["Date"].dt.dayofweek

# Sort historical records by date
historical_data = historical_data.sort_values("Date")

# Store predictions for each candidate window
validation_results = []

for window in candidate_windows:

    # Calculate rolling averages for each division and weekday
    historical_data["Rolling_Average"] = (
        historical_data.groupby(["DIVISION", "Weekday"])["Intervention_Count"]
        .transform(
            lambda values: values.rolling(
                window=window,
                min_periods=window
            ).mean()
        )
    )

    # Prepare historical averages for date-based matching
    historical_averages = historical_data[
        ["Date", "DIVISION", "Weekday", "Rolling_Average"]
    ].rename(columns={"Date": "Historical_Date"})

    # Match each forecast with the latest eligible historical average
    predictions = pd.merge_asof(
        validation_calendar.sort_values("Cutoff_Date"),
        historical_averages.sort_values("Historical_Date"),
        left_on="Cutoff_Date",
        right_on="Historical_Date",
        by=["DIVISION", "Weekday"],
        direction="backward"
    )

    # Keep the required prediction columns
    predictions = predictions.rename(
        columns={"Rolling_Average": "Baseline_Prediction"}
    )

    predictions["Window"] = window

    validation_results.append(
        predictions[["Date", "DIVISION", "Window", "Baseline_Prediction"]]
    )

# Combine predictions from all candidate windows
df_validation = pd.concat(validation_results, ignore_index=True)

# Display results
print("Validation predictions:", df_validation.shape)
print(
    "Missing predictions:",
    df_validation["Baseline_Prediction"].isna().sum()
)

display(df_validation.head(12))


Validation predictions: (35064, 4)
Missing predictions: 0


,Date,DIVISION,Window,Baseline_Prediction
0,2021-01-01,1,4,22.00
1,2021-01-01,2,4,56.50
2,2021-01-01,3,4,54.25
3,2021-01-01,4,4,63.00
4,2021-01-01,5,4,57.75
5,2021-01-01,6,4,32.75
6,2021-01-02,1,4,22.50
7,2021-01-02,2,4,42.50
8,2021-01-02,3,4,48.00
9,2021-01-02,4,4,48.75


### 2.15. Verify Optimized Baseline Predictions

The optimized implementation generated 35,064 predictions in approximately one second, compared with approximately two minutes using the original nested-loop implementation.

Both implementations produced the same dataset dimensions, no missing predictions, and identical results for the first 12 observations.

We will now compare additional predictions across different dates, divisions, and candidate windows to verify that the optimization preserves the original forecasting logic.

The comparison will include dates near the known missing data gap of December 31, 2024.

In [ ]:

# Select example dates for verification
test_dates = [
    "2021-01-01",
    "2022-06-15",
    "2023-09-18",
    "2024-01-19",
    "2024-12-30",
    "2024-12-31"
]

# Store comparison results
comparison_results = []

# Compare the original function with optimized predictions
for test_date in test_dates:
    for division in [1, 2, 3, 4, 5, 6]:
        for window in candidate_windows:

            original_prediction = predict_same_weekday_average(
                data=df_daily,
                division=division,
                forecast_date=test_date,
                window=window
            )

            matching_row = df_validation[
                (df_validation["Date"] == pd.Timestamp(test_date)) &
                (df_validation["DIVISION"] == division) &
                (df_validation["Window"] == window)
            ]

            optimized_prediction = matching_row["Baseline_Prediction"].iloc[0]

            comparison_results.append({
                "Date": test_date,
                "DIVISION": division,
                "Window": window,
                "Original": original_prediction,
                "Optimized": optimized_prediction,
                "Match": np.isclose(original_prediction, optimized_prediction)
            })

# Create a comparison DataFrame
df_comparison_check = pd.DataFrame(comparison_results)

print("Predictions compared:", len(df_comparison_check))
print("Matching predictions:", df_comparison_check["Match"].sum())
print("Non-matching predictions:", (~df_comparison_check["Match"]).sum())

# Display any differences
print("\nNon-matching observations:")
display(df_comparison_check[df_comparison_check["Match"] == False])


Predictions compared: 144
Matching predictions: 144
Non-matching predictions: 0

Non-matching observations:


,Date,DIVISION,Window,Original,Optimized,Match


**Methodological Observation**

The optimized baseline implementation was tested against the original forecasting function using 144 combinations of dates, operational divisions, and candidate windows.

All 144 predictions matched, with no differences identified. The tested dates included December 31, 2024, when actual intervention counts are unavailable.

The optimized implementation reduced the observed execution time from approximately two minutes to one second while preserving the forecasting results in the tested cases.

This supports using the optimized approach for chronological validation, although the comparison does not establish equivalence for every possible forecast.

### 2.16. Evaluate Candidate Baseline Windows

We will evaluate the four candidate baseline windows (4, 8, 13, and 26 same-weekday observations) using chronological validation from January 1, 2021, through December 31, 2024.

The predictions will be matched with actual intervention counts using the date and operational division.

December 31, 2024, will be excluded from error calculations because the actual intervention counts are missing.

Mean Absolute Error (MAE) will be used to compare forecasting accuracy, while Mean Error (bias) will indicate average overprediction or underprediction.

The reserved final evaluation period beginning January 1, 2025, will not be used for baseline selection.

In [ ]:

# Match validation predictions with actual intervention counts
df_validation_comparison = pd.merge(
    df_validation,
    df_daily,
    on=["Date", "DIVISION"],
    how="left"
)

# Count missing actual intervention values
missing_actuals = df_validation_comparison["Intervention_Count"].isna().sum()

print("Validation observations:", len(df_validation_comparison))
print("Missing actual counts:", missing_actuals)

# Exclude observations without known actual counts
df_validation_evaluation = df_validation_comparison.dropna(
    subset=["Intervention_Count", "Baseline_Prediction"]
).copy()

# Calculate prediction errors
df_validation_evaluation["Error"] = (
    df_validation_evaluation["Baseline_Prediction"]
    - df_validation_evaluation["Intervention_Count"]
)

# Calculate absolute errors
df_validation_evaluation["Absolute_Error"] = (
    df_validation_evaluation["Error"].abs()
)

# Calculate metrics for each candidate window
window_metrics = df_validation_evaluation.groupby("Window")[
    ["Absolute_Error", "Error"]
].mean()

# Rename metric columns
window_metrics = window_metrics.rename(
    columns={
        "Absolute_Error": "MAE",
        "Error": "Bias"
    }
)

print("\nValidation performance by window:")
display(window_metrics.round(3))

print("\nEvaluated predictions:", len(df_validation_evaluation))


Validation observations: 35064
Missing actual counts: 24

Validation performance by window:


,MAE,Bias
Window,,
4,9.203,-0.175
8,8.642,-0.301
13,8.480,-0.483
26,8.512,-0.893



Evaluated predictions: 35040


**Methodological Observation**

The chronological validation from 2021 through 2024 compared four same-weekday baseline windows using Mean Absolute Error (MAE) and Mean Error (bias).

The 13-observation window achieved the lowest overall MAE (8.480), followed closely by the 26-observation window (8.512). The 4-observation window had the highest MAE (9.203).

All candidate windows showed slightly negative bias, indicating average underprediction during the validation period.

These results identify the 13-observation window as the leading candidate based on overall MAE. However, the difference between the 13- and 26-observation windows is small, and division-level performance should be examined before confirming the baseline configuration.

### 2.17. Compare Baseline Windows by Division

We will compare the four candidate baseline windows separately for each operational division using chronological validation data from 2021 through 2024.

Mean Absolute Error (MAE) will be calculated for each combination of division and window.

This analysis will help determine whether the 13-observation window, which achieved the lowest overall validation MAE, also provides competitive forecasting performance across the six divisions.

The results will support the selection of a consistent baseline configuration without using the reserved final evaluation period.

In [ ]:

# Calculate MAE for each division and candidate window
division_window_mae = df_validation_evaluation.groupby(
    ["DIVISION", "Window"]
)["Absolute_Error"].mean()

# Display candidate windows as columns
division_window_mae = division_window_mae.unstack("Window")

# Display the MAE comparison
print("Validation MAE by division and window:")
display(division_window_mae.round(3))

# Identify the window with the lowest MAE for each division
best_window_by_division = division_window_mae.idxmin(axis=1)

print("\nLowest-MAE window by division:")
display(best_window_by_division)


Validation MAE by division and window:


Window,4,8,13,26
DIVISION,,,,
1,7.018,6.431,6.319,6.278
2,9.489,8.940,8.773,8.742
3,10.711,10.021,9.707,9.501
4,9.693,8.988,8.844,8.771
5,10.129,9.560,9.359,9.545
6,8.179,7.913,7.880,8.233



Lowest-MAE window by division:


DIVISION
1    26
2    26
3    26
4    26
5    13
6    13
dtype: int64

**Methodological Observation**

Division-level validation showed that the optimal same-weekday averaging window differs across SIM's operational divisions.

The 26-observation window achieved the lowest MAE for Divisions 1 through 4, while the 13-observation window performed best for Divisions 5 and 6.

Although the 26-observation window performed best in four divisions, the 13-observation window achieved the lowest overall validation MAE (8.480 versus 8.512).

These findings suggest that a single historical averaging window may not be optimal for every division. However, the 13-observation window remains the leading candidate for a common baseline configuration based on the overall validation criterion.

The stability of this result across validation years will be examined before confirming the final baseline window.

### 2.18. Compare Candidate Windows by Validation Year

We will calculate Mean Absolute Error (MAE) for each candidate baseline window separately for the years 2021, 2022, 2023, and 2024.

This analysis will help assess whether the overall validation results are consistent across years or influenced by specific periods.

The comparison will use the existing chronological validation predictions and will not introduce information from the reserved final evaluation period.

In [ ]:

# Extract the year from each forecast date
df_validation_evaluation["Year"] = (
    df_validation_evaluation["Date"].dt.year
)

# Calculate MAE for each year and window
year_window_mae = df_validation_evaluation.groupby(
    ["Year", "Window"]
)["Absolute_Error"].mean()

# Display windows as columns
year_window_mae = year_window_mae.unstack("Window")

# Display the yearly MAE comparison
print("Validation MAE by year and window:")
display(year_window_mae.round(3))

# Identify the lowest-MAE window for each year
best_window_by_year = year_window_mae.idxmin(axis=1)

print("\nLowest-MAE window by year:")
display(best_window_by_year)


Validation MAE by year and window:


Window,4,8,13,26
Year,,,,
2021,7.412,7.135,7.255,7.597
2022,8.335,7.855,7.789,7.826
2023,12.178,11.314,10.719,10.468
2024,8.887,8.265,8.159,8.156



Lowest-MAE window by year:


Year
2021     8
2022    13
2023    26
2024    26
dtype: int64

**Methodological Observation**

The yearly validation results show that the lowest-MAE baseline window varied across the four validation years.

The 8-observation window performed best in 2021, the 13-observation window in 2022, and the 26-observation window in 2023 and 2024.

Forecasting errors were notably higher in 2023 across all candidate windows, suggesting differences in forecasting difficulty over time. The cause of this variation has not been established.

Despite these yearly differences, the 13-observation window achieved the lowest overall MAE (8.480) across the complete 2021–2024 validation period.

The 13-observation window is therefore selected as the common baseline configuration based on the predefined overall MAE criterion. This selection does not imply that it will perform best in every division or future period.

### 2.19. Confirm the Selected Baseline

Based on chronological validation from 2021 through 2024, the 13-observation same-weekday average was selected as the common baseline configuration.

This window achieved the lowest overall validation MAE among the four candidates.

We will now retain the validation predictions generated with the selected window and calculate the corresponding MAE and Mean Error for each operational division.

The selected window will remain fixed during final evaluation. No information from the reserved 2025–2026 test period will be used to modify this configuration.

In [ ]:

# Define the selected baseline window
selected_window = 13

# Keep validation results for the selected window
df_selected_baseline = df_validation_evaluation[
    df_validation_evaluation["Window"] == selected_window
].copy()

# Calculate MAE and bias by division
selected_baseline_metrics = df_selected_baseline.groupby("DIVISION")[
    ["Absolute_Error", "Error"]
].mean()

# Rename metric columns
selected_baseline_metrics = selected_baseline_metrics.rename(
    columns={
        "Absolute_Error": "MAE",
        "Error": "Bias"
    }
)

# Display the selected baseline results
print("Selected baseline window:", selected_window)
print("Evaluated predictions:", len(df_selected_baseline))

print("\nSelected baseline performance by division:")
display(selected_baseline_metrics.round(3))

print("\nOverall validation MAE:", round(df_selected_baseline["Absolute_Error"].mean(), 3))
print("Overall validation Bias:", round(df_selected_baseline["Error"].mean(), 3))


Selected baseline window: 13
Evaluated predictions: 8760

Selected baseline performance by division:


,MAE,Bias
DIVISION,,
1,6.319,-0.337
2,8.773,-0.443
3,9.707,-0.597
4,8.844,-0.372
5,9.359,-0.365
6,7.880,-0.783



Overall validation MAE: 8.48
Overall validation Bias: -0.483


**Methodological Observation**

The 13-observation same-weekday average was confirmed as the selected baseline configuration based on chronological validation from 2021 through 2024.

The selected baseline achieved an overall MAE of 8.480 interventions across 8,760 evaluated division-day forecasts.

Division-level MAE ranged from 6.319 in Division 1 to 9.707 in Division 3. All six divisions showed slightly negative Mean Error values, indicating average underprediction during the validation period.

Although other windows performed better for certain divisions and years, the 13-observation window achieved the lowest overall validation MAE and will be retained as the common baseline configuration.

The selected window will remain fixed during the reserved final evaluation period.

### 2.20. Calculate Error Variance and Standard Deviation

We will calculate error variance and standard deviation for the selected 13-observation baseline during chronological validation.

Error variance measures the average squared difference between each forecast error and the mean error. Standard deviation expresses this variation in the original units of intervention counts.

The calculations will use population variance (`ddof=0`), consistent with the evaluation definition in the revised Business Problem Framing Report.

The metrics will be reported separately for each operational division.

In [ ]:

# Calculate error variance and standard deviation by division
error_variance = df_selected_baseline.groupby("DIVISION")["Error"].var(ddof=0)

error_std = df_selected_baseline.groupby("DIVISION")["Error"].std(ddof=0)

# Add the metrics to the selected baseline summary
selected_baseline_metrics["Error_Variance"] = error_variance
selected_baseline_metrics["Error_Std"] = error_std

# Display all four evaluation metrics
print("Selected baseline validation metrics by division:")
display(selected_baseline_metrics.round(3))


Selected baseline validation metrics by division:


,MAE,Bias,Error_Variance,Error_Std
DIVISION,,,,
1,6.319,-0.337,213.045,14.596
2,8.773,-0.443,183.406,13.543
3,9.707,-0.597,300.179,17.326
4,8.844,-0.372,172.451,13.132
5,9.359,-0.365,234.530,15.314
6,7.880,-0.783,110.771,10.525


**Methodological Observation**

Error variance and standard deviation were calculated for the selected 13-observation baseline using chronological validation data from 2021 through 2024.

Division 3 showed the highest error variance (300.179) and standard deviation (17.326), indicating the greatest dispersion of forecast errors around the mean error. Division 6 showed the lowest error variance (110.771) and standard deviation (10.525).

Division 1 achieved the lowest MAE (6.319), despite having a higher error standard deviation than several other divisions. This illustrates that average absolute error and error variability describe different aspects of forecasting performance.

These results complete the four technical evaluation metrics defined in the revised Business Problem Framing Report for the selected baseline during validation.

### 2.21. Save Baseline Validation Results

We will save the selected baseline's chronological validation predictions and division-level evaluation metrics.

These files will document the results of the 13-observation same-weekday baseline and support comparisons with more complex forecasting models in subsequent project stages.

The reserved final evaluation period from January 1, 2025, through September 24, 2026, has not been evaluated or used for baseline selection.

In [ ]:

# Define output file paths
predictions_file = "../../data/interim/modelling/SIM_Baseline_Validation_Predictions.csv"
metrics_file = "../../data/interim/modelling/SIM_Baseline_Validation_Metrics.csv"

# Save selected baseline predictions and errors
df_selected_baseline.to_csv(predictions_file, index=False)

# Save division-level evaluation metrics
selected_baseline_metrics.to_csv(metrics_file, index=True)

# Confirm saved files
print("Predictions saved to:", predictions_file)
print("Metrics saved to:", metrics_file)

print("\nSaved prediction records:", len(df_selected_baseline))
print("Saved division metrics:", len(selected_baseline_metrics))


Predictions saved to: ../../data/interim/modelling/SIM_Baseline_Validation_Predictions.csv
Metrics saved to: ../../data/interim/modelling/SIM_Baseline_Validation_Metrics.csv

Saved prediction records: 8760
Saved division metrics: 6


## 2.22. Stage 2 Conclusion — Baseline Forecasting

The Baseline Forecasting stage has been completed. A same-weekday averaging method was implemented and evaluated using chronological validation data from January 1, 2021, through December 31, 2024.

The baseline generated daily forecasts for each of SIM's six operational divisions using historical intervention counts available through D−3, consistent with the revised Business Problem Framing Report.

### Baseline Configuration

Four candidate historical averaging windows were evaluated: 4, 8, 13, and 26 same-weekday observations.

The **13-observation window** achieved the lowest overall validation Mean Absolute Error (MAE) and was selected as the common baseline configuration for all six divisions.

### Final Validation Summary

| Metric | Result |
|---|---:|
| Validation period | 2021-01-01 to 2024-12-31 |
| Candidate windows | 4, 8, 13, 26 |
| Selected window | 13 |
| Evaluated forecasts | 8,760 |
| Overall MAE | 8.480 |
| Overall Mean Error (Bias) | −0.483 |
| Divisions evaluated | 6 |
| Missing division-day targets excluded | 6 |

### Selected Baseline Performance by Division

| Division | MAE | Bias | Error Variance | Error Std |
|---|---:|---:|---:|---:|
| 1 | 6.319 | −0.337 | 213.045 | 14.596 |
| 2 | 8.773 | −0.443 | 183.406 | 13.543 |
| 3 | 9.707 | −0.597 | 300.179 | 17.326 |
| 4 | 8.844 | −0.372 | 172.451 | 13.132 |
| 5 | 9.359 | −0.365 | 234.530 | 15.314 |
| 6 | 7.880 | −0.783 | 110.771 | 10.525 |

### Key Methodological Decisions

- The baseline uses the average of recent valid observations from the same weekday, calculated separately for each division.
- All forecasts respect the D−3 historical information cutoff.
- The averaging window was selected using chronological validation data rather than the reserved final evaluation period.
- A common 13-observation window was selected based